# Historical Feature Engineering

Tujuan: Membangun **fitur historis** (*historical features*) yang merepresentasikan performa masing-masing tim sebelum pertandingan dimulai. Berbeda dengan *feature engineering* pada tahap sebelumnya yang hanya menggunakan informasi dasar seperti identitas tim, turnamen, dan status kandang, tahap ini mulai memanfaatkan riwayat pertandingan untuk menangkap kekuatan serta performa terkini setiap tim. Fitur-fitur yang dibangun diharapkan mampu memberikan informasi tambahan kepada model, misalnya mengenai konsistensi kemenangan, produktivitas mencetak gol, maupun kemampuan bertahan suatu tim. Seluruh fitur historis akan dihitung hanya menggunakan pertandingan yang telah terjadi sebelum pertandingan yang sedang diproses sehingga tetap merepresentasikan kondisi nyata (*real-world prediction*) dan terhindar dari *data leakage*. Output dari notebook ini adalah dataset baru yang berisi fitur baseline beserta fitur historis, sehingga siap digunakan pada proses pemodelan berikutnya.

## 1. Loading

Memuat dataset `baseline_features` yang telah disimpan pada tahap sebelumnya. 

In [1]:
import pandas as pd
import numpy as np

In [3]:
matches = pd.read_csv(
    "../data/processed/baseline_features.csv",
    parse_dates=["date"]
)

In [4]:
matches.head()

,date,home_team,away_team,tournament,neutral,match_result
0,1872-11-30,Scotland,England,Friendly,False,D
1,1873-03-08,England,Scotland,Friendly,False,H
2,1874-03-07,Scotland,England,Friendly,False,H
3,1875-03-06,England,Scotland,Friendly,False,D
4,1876-03-04,Scotland,England,Friendly,False,H


Dataset baseline berhasil dimuat dan akan digunakan sebagai dasar dalam membangun fitur historis. Seluruh pertandingan telah diurutkan secara kronologis sehingga informasi historis setiap tim dapat dihitung secara bertahap berdasarkan urutan waktu.

## 2. Konsep Historical Features

Pada tahap sebelumnya, model hanya memperoleh informasi dasar mengenai pertandingan, seperti identitas tim, jenis turnamen, dan status kandang. Meskipun informasi tersebut mampu memberikan kemampuan prediksi awal, model belum mengetahui bagaimana performa masing-masing tim sebelum pertandingan berlangsung.

Sebagai contoh, pertandingan berikut memiliki informasi dasar yang sama:

France vs Spain

Namun kenyataannya, kondisi kedua tim sebelum pertandingan dapat sangat berbeda.

Contoh:

France
5 pertandingan terakhir

W
W
D
W
W


Spain
5 pertandingan terakhir

L
D
W
L
D

Meskipun kedua pertandingan memiliki pasangan tim yang sama, peluang kemenangan tentu berbeda apabila salah satu tim sedang berada dalam performa yang jauh lebih baik.

Oleh karena itu, notebook ini membangun sejumlah fitur historis yang menggambarkan performa tim berdasarkan pertandingan-pertandingan sebelumnya. Seluruh perhitungan dilakukan secara kronologis sehingga setiap pertandingan hanya menggunakan informasi yang memang telah tersedia sebelum pertandingan tersebut dimainkan.

### Prinsip Utama

Dalam notebook ini, setiap fitur historis mengikuti prinsip berikut:
- Hanya menggunakan pertandingan yang terjadi sebelum pertandingan yang sedang diproses.
- Tidak menggunakan informasi hasil pertandingan yang sedang diprediksi.
- Seluruh perhitungan dilakukan secara kronologis berdasarkan tanggal pertandingan.
- Menghindari penggunaan informasi masa depan (*future information leakage*).
Dengan pendekatan ini, fitur yang dihasilkan dapat digunakan baik pada proses pelatihan model maupun pada prediksi pertandingan di dunia nyata.

## 3. Historical Feature Preparation
Kita harus melakukan persiapan terlebih dahulu.

### 3.1 Urutkan Data
Walaupun dataset hasil FE kemungkinan besar sudah urut, harus tetap divalidasi.

In [5]:
matches = (
    matches
    .sort_values("date")
    .reset_index(drop=True)
)

In [6]:
matches[["date", "home_team", "away_team"]].head()

,date,home_team,away_team
0,1872-11-30,Scotland,England
1,1873-03-08,England,Scotland
2,1874-03-07,Scotland,England
3,1875-03-06,England,Scotland
4,1876-03-04,Scotland,England


### 3.1 Insight

Dataset telah diurutkan berdasarkan tanggal pertandingan sehingga setiap fitur historis dapat dihitung secara kronologis. Langkah ini penting untuk memastikan bahwa seluruh perhitungan hanya memanfaatkan informasi dari pertandingan yang telah berlangsung sebelumnya.

### 3.2 Membuat Struktur Penyimpanan Riwayat Tim
Kita perlu tempat menyimpan histori tiap negara.

In [7]:
team_history = {}

Dictionary `team_history` akan digunakan sebagai tempat menyimpan riwayat pertandingan setiap tim. Struktur ini memungkinkan proses perhitungan fitur historis dilakukan secara bertahap seiring iterasi kronologis pada dataset.

### 3.3 Menentukan Informasi yang Disimpan
Satu pertandingan akan menyimpan informasi apa saja?

| Kolom           | Alasan                   |
| --------------- | ------------------------ |
| Result          | Hitung winrate           |
| Goals Scored    | Avg goals                |
| Goals Conceded  | Avg conceded             |
| Goal Difference | Goal difference form     |
| Date            | Validasi kronologi/debug |

Setiap kali sebuah pertandingan selesai diproses, informasi penting dari pertandingan tersebut akan disimpan ke dalam riwayat masing-masing tim. Informasi ini nantinya digunakan untuk menghitung berbagai fitur historis seperti persentase kemenangan, rata-rata gol, maupun selisih gol pada pertandingan berikutnya.